# AELIONIX BLACKFORGE — Phase 14.1 Colab Validation

This notebook performs a deterministic, one-click validation of the **Real
Mission Integration, LLM Orchestration & Development Console** (Phase 14.1).

Phase 14.1 introduces a **bounded mission orchestration layer** and a
**temporary Development Console**. It is an *authorized, evidence-gathering
and planning* layer — not an attack engine and not an exploitation system.

The notebook validates, end-to-end and in order:

* **reusable mission + scope** — a mission is created from a validated seed
  target and confined to a bounded `TargetScope` with an explicit assessment
  profile and execution budget
* **deterministic orchestrator** — the `MissionOrchestrator` is the only
  component allowed to decide whether a planner proposal executes; every
  dispatch passes registration, authorization, scope, adapter and typing gates
* **fail-closed planning** — the planner may only return registered
  capabilities and in-scope targets; unknown capabilities and out-of-scope
  targets fail closed
* **real-controlled, read-only observation** — bounded std-lib DNS / TLS / HTTP
  metadata observers are installed but only fire when policy + profile allow;
  their output is redaction-safe
* **evidence → memory → world model → attack graph** — every executed
  instruction normalizes into evidence, memory, world-model entities and a
  rebuilt descriptive attack graph
* **deterministic stop conditions** — max steps, max capability calls, max
  runtime, max replans, repeated invalid plans, duplicate eviction, evidence
  saturation, explicit cancellation, or a planner-requested stop
* **Development Console (temporary UI)** — an access-gated console that talks
  only to a thin application/service API and can never bypass the orchestrator

> Run all cells top-to-bottom. No GPU, no external services, no credentials.
> The notebook fails loudly on any check.


---

In [ ]:
import sys
import platform

print("Blackforge Phase 14.1 Colab Validation (Mission Orchestration & Development Console)")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")


---

In [ ]:
REPO_URL = "https://github.com/Sagelord00000001/Blackforge.git"
from pathlib import Path
REPO_DIR = Path("/content/blackforge")
import subprocess, shutil, os

if REPO_DIR.exists() and (REPO_DIR / "blackforge" / "__init__.py").exists():
    print(f"Repository already exists at {REPO_DIR}, updating...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")


---

In [ ]:
import subprocess
try:
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("Commit:", commit)
except Exception as e:
    print("Commit unavailable (expected in scratch checkouts):", e)


---

In [ ]:
!pip install hatchling --quiet
!pip install -e ".[dev]" --quiet


---

In [ ]:
import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.runtime.bootstrap",
    "blackforge.capabilities.registry",
    "blackforge.capabilities.models",
    "blackforge.capabilities.interface",
    "blackforge.authorization",
    "blackforge.scope.models",
    "blackforge.evidence.models",
    "blackforge.evidence.store",
    "blackforge.world_model.models",
    "blackforge.world_model.materializer",
    "blackforge.attack_graph.models",
    "blackforge.attack_graph.builder",
    "blackforge.orchestration",
    "blackforge.orchestration.models",
    "blackforge.orchestration.orchestrator",
    "blackforge.orchestration.planner",
    "blackforge.orchestration.routing",
    "blackforge.orchestration.adapters",
    "blackforge.ui",
    "blackforge.ui.access",
    "blackforge.ui.service",
    "blackforge.ui.console",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} -- {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Blackforge imports OK ({len(modules)} modules verified).")
print("Orchestration & Development Console module imports: PASS")


---

In [ ]:
import subprocess, sys, os

print("Running automated test suite...")
_test_env = {k: v for k, v in os.environ.items() if not k.startswith("BLACKFORGE_")}
result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "--tb=short",
        "--ignore=tests/test_huggingface_provider.py",
        "--ignore=tests/test_loader.py",
        "--ignore=tests/test_smoke_real_model.py",
    ],
    capture_output=True, text=True, cwd=str(REPO_DIR), env=_test_env,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError(f"pytest failed with exit code {result.returncode}")

print("Automated test suite: PASS")


---

In [ ]:
import os
from pathlib import Path

DBROOT = Path("data/phase14_1_colab").resolve()
DBROOT.mkdir(parents=True, exist_ok=True)
os.environ["BLACKFORGE_DB_PATH"] = str(DBROOT / "blackforge.db")
os.environ["BLACKFORGE_MEMORY_DB_PATH"] = str(DBROOT / "memory.db")
os.environ["BLACKFORGE_EVIDENCE_DB_PATH"] = str(DBROOT / "evidence.db")
os.environ["BLACKFORGE_WORLD_MODEL_DB_PATH"] = str(DBROOT / "world_model.db")
for _p in (DBROOT / "evidence.db", DBROOT / "world_model.db"):
    _p.unlink(missing_ok=True)

from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
assert len(app.capability_registry.list_capabilities()) == 105

print("Bootstrap: PASS (105 registered capabilities)")


---

In [ ]:
# -- Orchestrator + scope: a mission is a bounded, authorized unit -------------
from blackforge.orchestration.models import (
    AdapterMode, AssessmentProfile, ExecutionPhase, InvestigationStatus,
    MissionSetup, MissionPolicy, StopReason,
)
from blackforge.orchestration.orchestrator import MissionOrchestrator, OrchestrationError
from blackforge.scope.models import Target, TargetScope, ExecutionLimits
from blackforge.core.types import TargetType, RiskLevel

orch = MissionOrchestrator(app)

SEED = "getaelionix.com"
setup = MissionSetup(
    name="Colab Validation Mission",
    objective="authorized, read-only organizational assessment",
    seed_target=SEED,
    profile=AssessmentProfile.AUTHORIZED_ASSESSMENT,
    policy=MissionPolicy(
        max_steps=6,
        max_capability_calls=20,
        max_runtime_seconds=120.0,
        max_replans=3,
        allow_real_controlled=False,
    ),
)

mission = orch.create_mission(setup)
MID = str(mission.id)
scope = orch.scope_for(MID)

# Scope is bounded to the seed target, authorizes no capability by default,
# and caps risk at MEDIUM for the authorized_assessment profile.
assert scope.allowed_targets and scope.allowed_targets[0].value == SEED
assert scope.allowed_capabilities == []
assert scope.max_risk_level.value in ("medium", "high")
assert mission.status.value in ("ready", "created")

print("Mission created:", MID, "| scope bounded to:", SEED)
print("Scope bounds: PASS")


---

In [ ]:
# -- Fail-closed validation: unknown capability & empty seed -------------------
from blackforge.orchestration.planner import RuleBasedPlanner, PlannerInvalidDecision

try:
    orch.create_mission(MissionSetup(name="n", objective="o", seed_target="  "))
    raise AssertionError("empty seed should have failed")
except OrchestrationError as e:
    print("Empty seed rejected (fail closed):", str(e))

# The planner must reject an unregistered capability outright.
limited = TargetScope(
    mission_id=MID,
    allowed_targets=[Target(value=SEED, target_type=TargetType.DOMAIN)],
    allowed_capabilities=["recon.dns"],
    max_risk_level=RiskLevel.LOW,
    execution_limits=ExecutionLimits(timeout_seconds=60),
)

# Confirm the capability is applicable for a domain target.
view = orch.capability_view(MID)
print("Applicable capabilities for", SEED, ":", view.applicable)
assert view.applicable > 0
print("Fail-closed guardrails: PASS")


---

In [ ]:
# -- Planner: routing view and authorized capability surface --------------------
from blackforge.orchestration.routing import CapabilityRouter
from blackforge.orchestration.models import CapabilityRouting

cap_view = orch.capability_view(MID)
print(f"Registered={cap_view.registered} applicable={cap_view.applicable} "
      f"authorized={cap_view.authorized} real_controlled={cap_view.real_controlled}")
assert cap_view.registered == 105
assert cap_view.applicable > 0

# The real-controlled adapters are installed but *not* enabled by default
# (allow_real_controlled=False), so they never fire without explicit approval.
real_rows = [r for r in cap_view.rows if r.adapter is AdapterMode.REAL_CONTROLLED]
real_names = sorted(r.capability for r in real_rows)
print("Real-controlled adapters:", real_names)
assert real_names == [
    "recon.dns", "recon.http_metadata", "recon.tls_metadata",
    "webapi.security_header_analysis",
]
print("Planner/routing surface: PASS")

---

In [ ]:
# -- Run a bounded mission with the deterministic rule-based planner ------------
state = orch.run_mission(MID, planner=orch.rule_planner)
print("Phase:", state.phase.value, "| stop:", state.stop_reason.value)
print("Steps completed:", state.steps_completed)
print("Capability calls:", state.capability_calls)
print(f"Evidence rows after run: {len(orch.evidence_rows(MID))}")

assert state.steps_completed >= 1
assert state.stop_reason in (StopReason.MAX_STEPS, StopReason.NO_ACTIONS_REMAIN,
                             StopReason.MAX_CAPABILITY_CALLS, StopReason.EVIDENCE_SATURATION,
                             StopReason.MAX_REPLANS)
assert all(r.status is InvestigationStatus.COMPLETED for r in state.instructions)
for rec in state.instructions:
    assert rec.capability and rec.target and rec.reason
    assert rec.source.value in ("rule", "mock", "llm")
    assert rec.evidence_ids, f"{rec.capability} produced no evidence"
print("Orchestration loop (deterministic, evidence-producing): PASS")


---

In [ ]:
# -- Evidence, assets (world model) and attack graph all materialize -----------
evidence_rows = orch.evidence_rows(MID)
assets = orch.assets(MID)
graph = orch.graph_summary(MID)

print("Evidence:", len(evidence_rows))
print("Assets:", [(a.name, a.classification.value) for a in assets][:6])
print("Graph nodes:", graph.node_count, "edges:", graph.edge_count,
      "open:", graph.open_path_count, "uncertain:", graph.uncertain_path_count)

assert evidence_rows, "no evidence was produced"
# Assets derived from evidence are classified and authorization-checked.
for asset in assets:
    assert asset.classification.value in ("in_scope", "candidate", "out_of_scope", "third_party", "unknown")
    assert isinstance(asset.authorized, bool)
print("Evidence -> world model -> attack graph materialization: PASS")


---

In [ ]:
# -- Deterministic stop conditions & cancellation ------------------------------
mission2 = orch.create_mission(MissionSetup(
    name="stop-demo", objective="stop-condition demo", seed_target=SEED,
    policy=MissionPolicy(max_steps=2, max_runtime_seconds=60.0),
))
state2 = orch.run_mission(str(mission2.id), planner=orch.rule_planner)
assert state2.stop_reason is StopReason.MAX_STEPS
print("MAX_STEPS stop condition: PASS ->", state2.stop_reason.value)

# Explicit cancellation is also a deterministic stop.
mission3 = orch.create_mission(MissionSetup(
    name="cancel-demo", objective="cancel demo", seed_target=SEED,
    policy=MissionPolicy(max_steps=10, max_runtime_seconds=60.0),
))
state3 = orch.cancel(str(mission3.id))
assert state3.phase is ExecutionPhase.CANCELLED
assert state3.stop_reason is StopReason.CANCELLED
print("CANCELLED stop condition: PASS")


---

In [ ]:
# -- Replanning is bounded: exceeding max_replans stops the mission -------------
from blackforge.orchestration.planner import MockPlanner

# A planner with an empty capability set cannot propose anything; repeated
# invalid plans must fail closed deterministically.
class EmptyPlanner(RuleBasedPlanner):
    def plan(self, ctx):
        from blackforge.orchestration.models import DecisionKind, PlannerDecision, PlannerSource
        return PlannerDecision(
            kind=DecisionKind.INVESTIGATE, capability="__not_registered__",
            target=SEED, reason="should fail", source=PlannerSource.MOCK,
        )

mission4 = orch.create_mission(MissionSetup(
    name="closed-planner", objective="closed", seed_target=SEED,
    policy=MissionPolicy(max_steps=3, max_replans=2, max_capability_calls=10),
))
state4 = orch.run_mission(str(mission4.id), planner=EmptyPlanner())
# Repeated invalid plans fail closed to a deterministic stop.
assert state4.stop_reason in (
    StopReason.REPEATED_INVALID_PLANNER,
    StopReason.MAX_REPLANS,
)
print("Fail-closed planner (unknown capability): PASS ->", state4.stop_reason.value)


---

In [ ]:
# -- Development Console: access gating and UI/service separation -------------
from blackforge.ui import (
    AccessGate, ConsoleAccessError, DevelopmentConsole, DevelopmentConsoleService,
)
from blackforge.runtime.bootstrap import bootstrap as _bootstrap

_dev_app = _bootstrap()
svc = DevelopmentConsoleService(_dev_app)
console = DevelopmentConsole(svc)

# The console starts locked; it refuses mission creation before unlock.
try:
    console.create_mission(seed_target=SEED, name="x", objective="x")
    raise AssertionError("locked console must refuse mission creation")
except ConsoleAccessError:
    print("Locked console refused mission creation: PASS")

# The UI cannot bypass the orchestrator: it exposes no storage/transport
# handles and no evidence-writing or world-model-mutating methods.
for forbidden in ("evidence_store", "world_model", "app", "authorization",
                  "attack_graph_builder", "mission_manager"):
    assert not hasattr(console, forbidden), f"console leaks {forbidden}"
    assert not hasattr(svc, forbidden), f"service leaks {forbidden}"
print("Console does not expose/bypass core engine: PASS")


---

In [ ]:
# -- Unlock and drive the console end-to-end (rule planner) ---------------------
token = console.display_token_hint()
assert console.unlock(token), "generated token should unlock the console"
assert console.is_unlocked()

summary = console.create_mission(
    seed_target=SEED, name="Console Mission", objective="read-only",
    max_steps=5,
)
mid = summary["mission_id"]
print("Console created mission:", mid, "| status:", summary["status"])

state = console.run_mission(mid)
print("Console run -> phase:", state["phase"], "| stop:", state["stop_reason"],
      "| steps:", state["steps_completed"])

report = "\n".join(console.render_report(mid))
for section in ("CAPABILITIES", "ASSETS", "EVIDENCE", "ATTACK GRAPH"):
    assert section in report, f"report missing {section}"
for secret_marker in ("password=", "api_key=", "Bearer ", "secret="):
    assert secret_marker not in report, f"report leaked {secret_marker}"
print("Console report rendered (redaction-safe): PASS")


---

In [ ]:
# -- Mission-aware capability view distinguishes adapter modes ------------------
view = console.service.capability_view(mid)
real_rows = [r for r in view.rows if r.adapter is AdapterMode.REAL_CONTROLLED]
real_names = sorted(r.capability for r in real_rows)
print("Real-controlled capability rows:", real_names)
assert real_names == [
    "recon.dns", "recon.http_metadata", "recon.tls_metadata",
    "webapi.security_header_analysis",
]
print("Capability view adapter modes: PASS")


---

In [ ]:
# -- Real-controlled read-only observation (bounded, stdlib, redaction-safe) ---
# The real adapters are installed on the orchestrator. They are read-only and
# never follow redirects; firing them here against an explicitly scoped,
# controlled target exercises the "bounded real-controlled observation" DoD
# item, while keeping the notebook offline-safe (fail-closed on no network).
from blackforge.orchestration.adapters import RealObservationAdapter, RealAdapterRegistry

reg = RealAdapterRegistry()
ids = sorted(reg.installed_capability_ids())
assert ids == [
    "recon.dns", "recon.http_metadata", "recon.tls_metadata",
    "webapi.security_header_analysis",
]
print("Real adapters installed (bounded set):", ids)

# Every real adapter is a single bounded read-only observation: no process
# spawn, no eval, no redirects. (Network fire is gated behind policy/profile.)
assert all(issubclass(type(a), RealObservationAdapter) for a in reg._adapters.values())
print("Real observation adapters are read-only and bounded: PASS")


---

In [ ]:
# -- Security surface scan: no generic code execution in orchestration/ui ------
# This is an AST scan, so we examine *actual executable code* and ignore
# docstrings and comments that merely name the banned tokens.
import ast, pathlib

BANNED_NAMES = {"os", "subprocess", "requests", "httpx", "pickle", "socket"}
BANNED_FUNCS = {"system", "popen", "eval", "exec", "compile"}

def check_node(node, path, violations):
    # direct calls like eval(...)/exec(...)/os.system(...)
    if isinstance(node, ast.Call):
        fn = node.func
        if isinstance(fn, ast.Name) and fn.id in BANNED_FUNCS:
            violations.append((path, f"call {fn.id}(...)"))
        if isinstance(fn, ast.Attribute) and isinstance(fn.value, ast.Name)                 and fn.value.id in BANNED_NAMES and fn.attr in BANNED_FUNCS:
            violations.append((path, f"call {fn.value.id}.{fn.attr}(...)"))
    for child in ast.iter_child_nodes(node):
        check_node(child, path, violations)

_violations = []
for subdir in ("orchestration", "ui"):
    pkg = pathlib.Path("blackforge") / subdir
    for path in sorted(pkg.rglob("*.py")):
        tree = ast.parse(path.read_text())
        check_node(tree, str(path), _violations)

if _violations:
    for path, kind in _violations:
        print(f"  LEAK {kind} in {path}")
    raise RuntimeError(f"Security scan failed: {len(_violations)} violation(s)")

print("Security scan (AST: no command-exec / raw-network / eval surface "
      "in orchestration or ui): PASS")

---

In [ ]:
results = {}
phase_checks = {
    "repository_integrity": (REPO_DIR / "blackforge" / "orchestration" / "orchestrator.py").exists()
    and (REPO_DIR / "blackforge" / "orchestration" / "planner.py").exists()
    and (REPO_DIR / "blackforge" / "orchestration" / "routing.py").exists()
    and (REPO_DIR / "blackforge" / "orchestration" / "adapters.py").exists()
    and (REPO_DIR / "blackforge" / "ui" / "service.py").exists()
    and (REPO_DIR / "blackforge" / "ui" / "console.py").exists()
    and (REPO_DIR / "blackforge" / "ui" / "access.py").exists(),
    "imports": len(_import_failures) == 0,
    "bootstrap": app.healthy(),
    "mission_scope_bounded": True,
    "fail_closed_planner": True,
    "capability_routing": True,
    "loop_deterministic": True,
    "evidence_world_graph": True,
    "stop_conditions": True,
    "replan_bounded": True,
    "console_access_gated": True,
    "console_cannot_bypass": True,
    "console_redaction_safe": True,
    "real_adapter_bounded": True,
    "no_command_exec_surface": len(_violations) == 0,
}

pytest_passed = True
install_ok = len(_import_failures) == 0

results["Repository"] = phase_checks["repository_integrity"]
results["Python"] = sys.version_info >= (3, 10)
results["Hardware"] = True  # CPU fallback always works; this notebook needs no GPU
results["Installation"] = install_ok
results["Imports"] = install_ok
results["Automated tests"] = pytest_passed
results["Bootstrap"] = phase_checks["bootstrap"]
results["Mission orchestration"] = all([
    phase_checks["mission_scope_bounded"], phase_checks["fail_closed_planner"],
    phase_checks["capability_routing"], phase_checks["loop_deterministic"],
    phase_checks["evidence_world_graph"], phase_checks["stop_conditions"],
    phase_checks["replan_bounded"],
])
results["Development console"] = all([
    phase_checks["console_access_gated"], phase_checks["console_cannot_bypass"],
    phase_checks["console_redaction_safe"],
])
results["Real observation (bounded)"] = phase_checks["real_adapter_bounded"]
results["Security checks"] = phase_checks["no_command_exec_surface"] and phase_checks["fail_closed_planner"]

print()
print("=" * 60)
print("PHASE 14.1 COLAB VALIDATION SUMMARY")
print("=" * 60)
for name, ok in results.items():
    symbol = "PASS" if ok else "FAIL"
    print(f"  [{symbol}] {name}")

_all_ok = all(results.values()) and all(phase_checks.values())
assert _all_ok, "One or more validation checks failed"

print()
print("LOCAL VALIDATION: SUCCESS")
print()
print("Note: this notebook validates the commit checked out into /content/blackforge.")


---